In [ ]:
# test-file-5.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline

In [ ]:
# specifying the path to the csv file that needs to be imported (the overlap.csv)
input_file_path = "/mnt/d/code/phd/image-analysis/synapse-counting/output_data/overlap.csv"

df = pd.read_csv(input_file_path)
print(df.head(3))

In [ ]:
# transforming the df into the right format

# dropping the unneccesarry columns (vlgut1 threshold and psd95_threshold)
df_drop = df.drop(['vglut1_threshold', 'psd95_threshold'], axis=1)

# adding an extra element to the first column so that each observation is unique
df_drop["image file name"] = df_drop["image file name"] + "_" + (df_drop.groupby("image file name").cumcount() + 1).astype(str)

# pivoting the df into longer format
df_melted = pd.melt(df_drop, id_vars = ["image file name"], value_vars=["overlap_um2", "overlap_um2_rot"],
                    var_name="condition", value_name="overlap (um2)")

# replacing the names for the condition
df_melted["condition"] = df_melted["condition"].apply(lambda x: "rotated" if "overlap_um2_rot" in x else "actual")

# sorting the df based on the first column
df_sorted = df_melted.sort_values('image file name')

# have column that takes the hippocampal layer
df_sorted['hippocampal layer'] = df_sorted['image file name'].apply(lambda x: ' '.join(x.split('_')[-3:-1]))


print(df_sorted)

In [ ]:
# creating the plot
p = sns.stripplot(y="overlap (um2)", x="hippocampal layer", hue = "condition",
                        data = df_sorted,
                        jitter = False,
                        dodge = True,
                        marker = "o",
                        alpha = 0.5)

# sns.pointplot(y="overlap (um2)", x="hippocampal layer", hue = "condition",  
#               data=df_sorted, dodge=0.2, join=False, palette="dark",  
#               markers="_", scale=0.75, ax=p)

sns.boxplot(y="overlap (um2)", x="hippocampal layer", hue = "condition",  
              data=df_sorted, dodge=0.2,palette="dark",  
              ax=p)

In [ ]:
# # exporting the plot as png
# output_folder = "/mnt/d/code/phd/image-analysis/synapse-counting/output_data/"

# output_file = os.path.join(output_folder, "dotplot.png")
# plot = p.get_figure()
# plot.savefig(output_file, dpi = 300)

In [ ]:
# 

In [ ]:
# Separate the data for actual and rotated conditions
actual_data = df_sorted[df_sorted["condition"] == "actual"]
rotated_data = df_sorted[df_sorted["condition"] == "rotated"]

# Create the plot
plt.scatter(np.zeros(len(actual_data)), actual_data["overlap (um2)"], label="actual")
plt.scatter(np.ones(len(rotated_data)), rotated_data["overlap (um2)"], label="rotated")

# Plot the lines connecting points
for i in range(len(actual_data)):
    plt.plot([0, 1], [actual_data.iloc[i]["overlap (um2)"], rotated_data.iloc[i]["overlap (um2)"]], c='k')

plt.xticks([0, 1], ['actual', 'rotated'])
plt.ylabel("overlap (um2)")
plt.legend()

plt.show()